In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "litsearch"

In [2]:
from src.llm.factory import build_llm_client
from src.llm.cache import LLMCache
from pathlib import Path
from src.config import CACHE_DIR

DEEPSEEK_API_KEY = mlc_tools.get_secret("DeepseekAPI_GY")

# создать LLM client (deepseek)
llm_client = build_llm_client(
    provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    temperature=0.0,
    max_tokens=2048,
)

# создать кэш (TTL можно указать в секундах или None)
cache = LLMCache(cache_dir=CACHE_DIR, ttl=None)


In [3]:
def make_cached_llm_callable(llm_client, cache, meta_template=None):
    """
    Возвращает функцию (prompt)->str, которая сначала проверяет кэш, 
    затем вызывает llm_client.generate и сохраняет результат.
    meta_template — словарь, куда попадают model/temperature/max_tokens/task и т.д.
    """
    
    def wrapped(prompt: str, task: str = None):
        meta = dict(meta_template or {})
        if task:
            meta["task"] = task
        # add model params to meta for safety
        meta["model"] = getattr(llm_client, "model", None)
        meta["temperature"] = getattr(llm_client, "temperature", None)
        meta["max_tokens"] = getattr(llm_client, "max_tokens", None)
        
        # внутри wrapped: if cached: cache_hits["hits"] += 1 else cache_hits["miss"] += 1


        # cache.get_or_call expects call_fn(prompt) -> str
        return cache.get_or_call(prompt, lambda p: llm_client.generate(p), meta=meta)
    return wrapped

# создаём callable
llm_cached = make_cached_llm_callable(llm_client, cache, meta_template={"project":"corank"})


In [4]:
from src.rerank.coarse import run_coarse_reranking

# pass wrapper: it must accept single argument prompt (we already have that)
run_coarse_reranking(
    ie_path=DATA_DIR / "ie_documents.jsonl",
    queries_path=DATA_DIR / "queries.jsonl",
    output_path=DATA_DIR / "coarse_rerank.jsonl",
    llm_call_fn=lambda prompt: llm_cached(prompt, task="coarse"),
    top_k=20,
    max_queries = 3
)


HTTPError: 402 Client Error: Payment Required for url: https://api.deepseek.com/v1/chat/completions

In [ ]:
from src.rerank.fine import run_fine_reranking

run_fine_reranking(
    coarse_path=DATA_DIR / "coarse_rerank.jsonl",
    queries_path=DATA_DIR / "queries.jsonl",
    corpus_path=DATA_DIR / "corpus.jsonl",
    output_path=DATA_DIR / "fine_rerank.jsonl",
    llm_call_fn=lambda prompt: llm_cached(prompt, task="fine"),
    max_docs = 20
)


In [ ]:
# для IE мини-прогона:
from src.ie.extract_from_candidates import run_ie_on_candidates
run_ie_on_candidates(
    candidates_path=DATA_DIR / "candidates.jsonl",
    corpus_path=DATA_DIR / "corpus.jsonl",
    output_path=DATA_DIR / "ie_documents_mini.jsonl",
    llm_call_fn=lambda p: llm_cached(p, task="ie"),
    max_queries=10   # если функция поддерживает этот аргумент
)


In [ ]:
results = evaluate_runs(qrels, runs, metrics)
results


In [ ]:
!rm -rf data/cache/llm